<a href="https://colab.research.google.com/github/Ridzz110/Flow-research-applied-ai-engineering/blob/main/Few_shot_learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers accelerate datasets torch

In [ ]:
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

print("Model loaded!")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Model loaded!


In [ ]:
dataset = load_dataset("stanfordnlp/imdb")

train = dataset["train"]
test = dataset["test"]

label_names = {
    0: "Negative",
    1: "Positive"
}

# Small fixed dataset for the experiment
demo_data = train.select(range(20))
test_data = test.select(range(20))

print("Demo examples:", len(demo_data))
print("Test examples:", len(test_data))

Demo examples: 20
Test examples: 20


In [ ]:
def get_examples(data):
    examples = []

    for item in data:
        examples.append({
            "text": item["text"][:300],
            "label": label_names[item["label"]]
        })

    return examples


demo_examples = get_examples(demo_data)
test_examples = get_examples(test_data)

print(demo_examples[0])

{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really h', 'label': 'Negative'}


In [ ]:
def create_prompt(demos, test_text):

    prompt = """Classify the sentiment of the following text as Positive or Negative.

"""

    for i, example in enumerate(demos, 1):
        prompt += f"""Example {i}:
Text: {example['text']}
Sentiment: {example['label']}

"""

    prompt += f"""Now classify this text:

Text: {test_text}
Sentiment:"""

    return prompt

In [ ]:
def predict_sentiment(prompt):
    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=5,
            do_sample=False
        )

    # Only decode newly generated tokens
    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

    return response

In [ ]:
def clean_prediction(prediction):

    prediction = prediction.lower()

    if "positive" in prediction:
        return "Positive"

    elif "negative" in prediction:
        return "Negative"

    else:
        return "Unknown"

In [ ]:
prompt = create_prompt(
    demo_examples[:2],
    test_examples[0]["text"]
)

print("PROMPT:")
print(prompt)

response = predict_sentiment(prompt)

print("\nRAW MODEL RESPONSE:")
print(response)

print("\nCLEANED RESPONSE:")
print(clean_prediction(response))

PROMPT:
Classify the sentiment of the following text as Positive or Negative.

Example 1:
Text: I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really h
Sentiment: Negative

Example 2:
Text: "I Am Curious: Yellow" is a risible and pretentious steaming pile. It doesn't matter what one's political views are because this film can hardly be taken seriously on any level. As for the claim that frontal male nudity is an automatic NC-17, that isn't true. I've seen R-rated films with male nudity
Sentiment: Negative

Now classify this text:

Text: I love sci-fi and am willing to put up with a lot. Sci-fi movies/TV are usually underfunded, under-appreciated and misunderstood. I tried to like this, I really did, but it is to good TV sci-fi as Babylon 5 is to

In [ ]:
def run_count_experiment(k):

    demos = demo_examples[:k]

    results = []

    for item in test_examples:

        prompt = create_prompt(
            demos,
            item["text"]
        )

        response = predict_sentiment(prompt)

        prediction = clean_prediction(response)

        results.append({
            "Actual": item["label"],
            "Predicted": prediction
        })

    df = pd.DataFrame(results)

    accuracy = (
        df["Actual"] == df["Predicted"]
    ).mean()

    return accuracy, df

In [ ]:
counts = [0, 1, 2, 4, 8]

count_results = []

for k in counts:

    print(f"Running K = {k}")

    accuracy, _ = run_count_experiment(k)

    count_results.append({
        "Number of Examples (K)": k,
        "Accuracy (%)": round(accuracy * 100, 2)
    })


count_results_df = pd.DataFrame(count_results)

count_results_df

Running K = 0
Running K = 1
Running K = 2
Running K = 4
Running K = 8


,Number of Examples (K),Accuracy (%)
0,0,85.0
1,1,80.0
2,2,80.0
3,4,85.0
4,8,85.0


In [ ]:
order_a = demo_examples[:4]

order_b = [
    demo_examples[3],
    demo_examples[2],
    demo_examples[1],
    demo_examples[0]
]

order_c = [
    demo_examples[1],
    demo_examples[3],
    demo_examples[0],
    demo_examples[2]
]

orders = {
    "Order A": order_a,
    "Order B": order_b,
    "Order C": order_c
}

In [ ]:
order_results = []

for order_name, demos in orders.items():

    print(f"Running {order_name}")

    correct = 0

    for item in test_examples:

        prompt = create_prompt(
            demos,
            item["text"]
        )

        response = predict_sentiment(prompt)

        prediction = clean_prediction(response)

        if prediction == item["label"]:
            correct += 1

    accuracy = correct / len(test_examples)

    order_results.append({
        "Order": order_name,
        "Accuracy (%)": round(accuracy * 100, 2)
    })


order_results_df = pd.DataFrame(order_results)

order_results_df

Running Order A
Running Order B
Running Order C


,Order,Accuracy (%)
0,Order A,85.0
1,Order B,75.0
2,Order C,85.0


In [ ]:
def create_format_prompt(demos, test_text, format_type):

    # Format 1: Input / Output
    if format_type == "Input-Output":

        prompt = """Classify the sentiment as Positive or Negative.

"""

        for example in demos:
            prompt += f"Input: {example['text']}\n"
            prompt += f"Output: {example['label']}\n\n"

        prompt += f"Input: {test_text}\n"
        prompt += "Output:"

    # Format 2: Question / Answer
    elif format_type == "Question-Answer":

        prompt = """Determine the sentiment of each text.

"""

        for example in demos:
            prompt += f"Q: {example['text']}\n"
            prompt += f"A: {example['label']}\n\n"

        prompt += f"Q: {test_text}\n"
        prompt += "A:"

    # Format 3: Natural language
    elif format_type == "Natural Language":

        prompt = ""

        for example in demos:
            prompt += (
                f'"{example["text"]}" '
                f'has a {example["label"]} sentiment.\n'
            )

        prompt += (
            f'\n"{test_text}" has a'
        )

    return prompt

In [ ]:
for format_type in [
    "Input-Output",
    "Question-Answer",
    "Natural Language"
]:

    print("\n" + "=" * 60)
    print(format_type)
    print("=" * 60)

    print(
        create_format_prompt(
            demo_examples[:4],
            test_examples[0]["text"],
            format_type
        )
    )


Input-Output
Classify the sentiment as Positive or Negative.

Input: I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really h
Output: Negative

Input: "I Am Curious: Yellow" is a risible and pretentious steaming pile. It doesn't matter what one's political views are because this film can hardly be taken seriously on any level. As for the claim that frontal male nudity is an automatic NC-17, that isn't true. I've seen R-rated films with male nudity
Output: Negative

Input: If only to avoid making this type of film in the future. This film is interesting as an experiment but tells no cogent story.<br /><br />One might feel virtuous for sitting thru it because it touches on so many IMPORTANT issues but it does so without any discernable motive. The

In [ ]:
formats = [
    "Input-Output",
    "Question-Answer",
    "Natural Language"
]

format_results = []

for format_type in formats:

    print(f"Running {format_type}")

    correct = 0

    for item in test_examples:

        prompt = create_format_prompt(
            demo_examples[:4],
            item["text"],
            format_type
        )

        response = predict_sentiment(prompt)

        prediction = clean_prediction(response)

        if prediction == item["label"]:
            correct += 1

    accuracy = correct / len(test_examples)

    format_results.append({
        "Format": format_type,
        "Accuracy (%)": round(accuracy * 100, 2)
    })


format_results_df = pd.DataFrame(format_results)

format_results_df

Running Input-Output
Running Question-Answer
Running Natural Language


,Format,Accuracy (%)
0,Input-Output,85.0
1,Question-Answer,85.0
2,Natural Language,85.0


In [ ]:
print("EXPERIMENT 1 — NUMBER OF EXAMPLES")
display(count_results_df)

print("\nEXPERIMENT 2 — EXAMPLE ORDER")
display(order_results_df)

print("\nEXPERIMENT 3 — PROMPT FORMAT")
display(format_results_df)

EXPERIMENT 1 — NUMBER OF EXAMPLES


,Number of Examples (K),Accuracy (%)
0,0,85.0
1,1,80.0
2,2,80.0
3,4,85.0
4,8,85.0



EXPERIMENT 2 — EXAMPLE ORDER


,Order,Accuracy (%)
0,Order A,85.0
1,Order B,75.0
2,Order C,85.0



EXPERIMENT 3 — PROMPT FORMAT


,Format,Accuracy (%)
0,Input-Output,85.0
1,Question-Answer,85.0
2,Natural Language,85.0
